In [66]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict

In [67]:
load_dotenv()  # Load environment variables from .env file

True

In [68]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [69]:
# create a state 
class LLMState(TypedDict):
    question: str
    answer: str


In [70]:
def llm_qa(state: LLMState) -> LLMState:
#    extract the question from the state
    question = state["question"]
    
# form the prompt for the LLM
    prompt = f"Answer the following question: {question}"

# ask that question to the LLM
    answer = model.invoke(prompt).content
# update the state with the answer
    state["answer"] = answer
    return state

In [71]:
# create a state graph 
graph = StateGraph(LLMState)

# add nodes to the graph
graph.add_node('llm-qa', llm_qa)

# add edges to the graph
graph.add_edge(START, 'llm-qa')
graph.add_edge('llm-qa', END)

# compile the graph into a workflow
workflow = graph.compile()

In [72]:
# execute the workflow with an initial state
initial_state = {'question': 'What is the capital of India?'}
final_state = workflow.invoke(initial_state)
print(final_state)  # Output: {'question': 'What is the capital of India?', 'answer': 'The capital of India is New Delhi.'}

{'question': 'What is the capital of India?', 'answer': 'The capital of India is **New Delhi**.'}
